# 六足ロボットの円柱押し：仮説から比較実験・考察まで

この教材では、**実際に学習した2条件×3 seedの完成例**を読み、自分の比較実験へ進みます。
調べるのは「円柱を目標へ近づけたときの報酬を20から40にすると、成功率が上がるか」です。

1. 「ランタイム → ランタイムのタイプを変更」で **T4 GPU** を選びます。
2. 上から実行し、まず完成例の仮説・条件・学習曲線・評価・動画・考察を読みます。
3. Step 8以降で、報酬を変更して自分の実験を行います。

完成例の結果はリポジトリに保存済みです。読むために長い学習をやり直す必要はありません。
自分の実験は最初は短い操作練習、`PROFILE="research"`で完成例と同じ本学習になります。

## Step 0 — 教材を準備する

GitHubから教材を取得し、必要なライブラリを準備します。
再実行時は取得済み教材を使います。更新版は新しいセッションで開いてください。

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/hashimoto-robotics-lab/hexapod-transport-rl.git"
REVISION = "main"
PROJECT_DIR = Path("/content/hexapod_transport_rl")

if not PROJECT_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REVISION,
        REPOSITORY, str(PROJECT_DIR),
    ], check=True)

sys.path.insert(0, str(PROJECT_DIR / "tools"))
from colab_setup import prepare_colab

prepare_colab(PROJECT_DIR)

## Step 1 — ロボットが何を見て、何をするかを理解する

1つの世界に、18関節の六足ロボット1台・円柱・目標があります。
円柱は半径0.25 m・高さ0.25 m・質量2 kgです。
ロボットは円柱から0.65〜1.00 mの周囲、向きは全周からランダムに開始します。
目標は円柱から0.30〜0.60 mの全方向からランダムに決めます。
**位置誤差5 cm未満、円柱速度0.06 m/s未満を0.4秒保持**すると成功。制限時間は40秒です。

方策は0.2秒ごとに、停止＋世界座標の8方向から1つを選びます。
選んだ方向は固定した歩行モデルを通じて18関節へ伝わり、接触によって円柱が動きます。
方向は機体の前後ではなく世界のXYです。歩行モデルが移動方向へ機体を旋回させます。
PPOが関節角を直接選ぶわけではなく、歩行モデルはこの実験では再学習しません。

In [ ]:
import json
from dataclasses import replace
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, Markdown, Video, display

from hexapod_transport_rl import (
    ContinuousVelocity, CylinderPushTask, DiscreteDirections, ObservationTerm,
    PPOSettings, compare, load_task_file, train,
)

STUDY_DIR = PROJECT_DIR / "results/reward_study"
protocol = json.loads((STUDY_DIR / "protocol.json").read_text())
summary = json.loads((STUDY_DIR / "summary.json").read_text())
display(Image(filename=str(STUDY_DIR / "task.png"), width=650))

薄緑の円は円柱を置きたい形と位置、中心の小さい円は5 cmの位置許容範囲を示します。
図の`cargo_to_robot`は**円柱からロボットへ**、`cargo_to_goal`は**円柱から目標へ**向かう世界XYベクトルです。
完成例の方策には次の14成分を渡します。ActorとCriticは同じ観測を使います。

| 入力の順序 | 意味 | 方策へ渡す値 |
|---|---|---|
| 0〜1 | 円柱からロボットへのXY | 距離[m] ÷ 0.5 |
| 2〜3 | 円柱から目標へのXY | 距離[m] ÷ 0.5 |
| 4〜5 | 互換性のための固定値 | `[0, 1]` |
| 6〜7 | ロボットの世界XY速度 | 速度[m/s] ÷ 0.2 |
| 8〜9 | 円柱の世界XY速度 | 速度[m/s] ÷ 0.2 |
| 10 | 互換性のための固定値 | `0` |
| 11〜12 | ロボットの向き | `[sin(yaw), cos(yaw)]` |
| 13 | ロボットの旋回速度 | rad/s |

`num_envs=256`は、ロボット256台で1つの円柱を押す意味ではありません。
独立した256個の世界から並列に経験を集めます。

## Step 2 — 結果を見る前に仮説を立てる

**完成例の仮説：** 円柱を目標へ近づける報酬の重みを20から40にすると、押す行動を選びやすくなり、同じ学習量で成功率が上がり、最終距離が小さくなる。

逆に、目標付近でも押し続けると、通り過ぎて停止条件を満たしにくくなる可能性もあります。
まず自分の予想と理由を1文で書いてから、次の結果を見てください。

標準報酬は、目標距離を`d`、回り込みコストを`a`、1ステップを`dt=0.2秒`として、

`20 × (前のd − 今のd) + 2 × (前のa − 今のa) − (4d + a + 0.02)dt + 10 × 成功 − 10 × 失敗`

です。変更条件は、先頭の**20だけを40**にします。
回り込みコストは円柱の表面までの距離と、目標の反対側へ回り込む角度をまとめた量です。
報酬を変えても成功条件は変えません。

In [ ]:
base = CylinderPushTask()
changed = replace(base, rewards={**base.rewards, "goal_progress": 40.0})

display(pd.DataFrame({
    "標準": {name: term.weight for name, term in base.rewards.items()},
    "変更": {name: term.weight for name, term in changed.rewards.items()},
}))

## Step 3 — 比較する条件を固定する

この完成例は、両条件とも**ランダムな初期重みから**学習しています。
同じseedの組はActor・Criticの初期重みも同じです。
観測、行動、初期状態分布、物理、PPO、学習サンプル数を揃え、報酬の重み1つだけを変更しました。
回転対称化は両条件とも使いません。

検証はseed 131000の256配置で行い、その結果だけで`best.pt`を選びます。
目標方向・開始位置・向きの区分ごとの最低成功率を優先し、次に全体の成功数と距離を比べる、両条件共通の基準です。
そのため、全体成功率が最も高い時点を選ぶとは限りません。
最終評価は別のseed 240000の256配置です。最終評価の結果で重みや良い学習seedを選び直しません。
`steps`は全並列世界の上位行動数の合計です。物理計算の細かい時間刻みの数とは異なります。

In [ ]:
display(pd.DataFrame({
    "固定した項目": ["学習seed", "各seedの学習量", "並列世界数", "1回の収集長", "最終評価回数", "変更した項目"],
    "値": [str(protocol["training_seeds"]), f"{protocol['training']['steps']:,} samples",
           str(protocol["training"]["num_envs"]), str(protocol["training"]["horizon"]),
           str(protocol["evaluation"]["episodes"]), "goal_progress: 20 → 40"],
}))

## Step 4 — 学習曲線を読む

左は共通の検証配置での成功率、右は終了時の目標距離です。
細線は学習seedごとの結果、太線と帯は3 seedの平均と標準偏差です。
横軸0には学習前の方策も含まれています。点は検証で選ばれた方策の時点です。

**読むポイント：** 学習が進むほど改善しているか、途中で悪化していないか、seedによる差が大きいか。
報酬の重みが異なるため、総報酬の大小だけで条件の優劣を決めません。

In [ ]:
display(Image(filename=str(STUDY_DIR / "learning_curves.png"), width=950))
learning_curve = pd.read_csv(STUDY_DIR / "learning_curve.csv")
display(learning_curve.tail(6))

## Step 5 — 最終評価とばらつきを確認する

各行は、1つの条件・1つの学習seedの結果です。
`baseline`は標準、`progress40`は重み40。成功率は0〜1（0.1なら10%）、距離はm、時間は秒です。
成功率を主な指標、終了時の目標距離を補助指標として比較します。
3 seedの標準偏差は学習によるばらつきです。評価エピソード数を増やしても、学習seedが3つであることは変わりません。

In [ ]:
results = pd.read_csv(STUDY_DIR / "runs.csv")
display(results[["condition", "seed", "successes", "episodes", "success_rate", "mean_distance_m", "selected_steps", "mean_episode_seconds"]])
display(pd.DataFrame(summary["conditions"]).T[["mean_success_rate", "std_success_rate", "mean_distance_m", "mean_training_seconds"]])
display(Image(filename=str(STUDY_DIR / "paired_results.png"), width=850))

線は同じ学習seed同士を結びます。すべてのseedで同じ方向に変化しているか確認してください。
`selected_steps`は検証で選ばれた学習時点です。学習の最後と一致するとは限りません。
実行時間はRTX A6000の共用ワークステーションで測った値で、Colab T4での所要時間ではありません。

**問い：** 平均だけを見る場合と、3本の線を見る場合で、結論の強さは変わりますか？

## Step 6 — 同じ初期配置の動画を見る

動画は、結果を見る前に決めた学習seed42・動画seed250000の組です。
良い動画だけを選んだものではありません。
円柱に近づくか、目標の反対側へ回り込むか、目標付近で止まるかを観察してください。

動画はCPUのMuJoCo、表の数値はGPUのMuJoCo Warpです。
計算方法の違いがあるため、動画1本の成否を表の成功数として数えません。

In [ ]:
for condition in ("baseline", "progress40"):
    print(condition)
    display(Video(str(STUDY_DIR / "videos" / f"{condition}.mp4"), embed=True, width=640))

## Step 7 — 完成例の考察と照らし合わせる

ここまでの図・表・動画から、自分の結論を先に1文で書いてください。
その後で完成例の考察を読み、**数値から分かる事実**と**理由の解釈**を分けて確認します。

In [ ]:
display(Markdown((STUDY_DIR / "discussion.md").read_text()))

## Step 8 — 自分の実験を設計する

次は1項目を自分で変更します。まず以下をMarkdownセルへ書きます。

- 仮説：何を変えると、どの指標が、なぜ変わるか。
- 変更する1項目：例 `goal_progress: 20 → 10`。
- 固定するもの：観測・行動・物理・学習量・PPO・学習seed・評価方法。
- 判定する指標：成功率、最終距離、seed間のばらつき。

下の`PROFILE="demo"`は、2秒・32サンプルの操作練習です。学習効果の結論には使いません。
`"research"`で完成例と同じ40秒・2,621,440サンプルの本学習に切り替わります。
本学習ではGPUを使い、時間に余裕を持って実行してください。
Step 9は1 seedの練習、Step 10が3 seedの比較本体です。Step 11の独自関数は発展課題です。
保存先には時刻を付けるので、値を変えてセルを再実行できます。

In [ ]:
PROFILE = "demo"  # "research" で完成例と同じ学習条件
MY_HYPOTHESIS = "進捗の重みを40にすると、同じ学習量で成功率が上がる"
PROGRESS_WEIGHT = 40.0  # 自分の仮説に合わせて変更

if PROFILE == "demo":
    my_base = replace(base, config=replace(base.config, episode_seconds=2.0))
    options = dict(steps=32, num_envs=2, horizon=8, validation_envs=2,
                   validate_every=1, settings=PPOSettings(epochs=1, minibatch_size=32))
    seeds, evaluation_episodes = (42, 43), 4
else:
    my_base = base
    options = {**protocol["training"], "settings": PPOSettings(**protocol["ppo"])}
    seeds, evaluation_episodes = tuple(protocol["training_seeds"]), protocol["evaluation"]["episodes"]

my_task = replace(my_base, rewards={**my_base.rewards, "goal_progress": PROGRESS_WEIGHT})
WORK_DIR = PROJECT_DIR / "runs" / datetime.now().strftime("tutorial_%Y%m%d_%H%M%S_%f")
WORK_DIR.mkdir(parents=True)
plan = {
    "hypothesis": MY_HYPOTHESIS, "profile": PROFILE,
    "changed_parameter": "goal_progress", "baseline_weight": 20.0,
    "changed_weight": PROGRESS_WEIGHT, "training_seeds": list(seeds),
    "evaluation_seed": 340000,
}
(WORK_DIR / "experiment_plan.json").write_text(json.dumps(plan, ensure_ascii=False, indent=2))
print(MY_HYPOTHESIS)
print("保存先:", WORK_DIR)

まず`reset`・`step`で、観測と報酬を確認できます。
`env.state`は報酬の設計で使う名前付き状態、`observation`は方策に渡す入力です。

In [ ]:
env = my_base.make_env(num_envs=1, backend="cpu", render_mode="rgb_array")
observation = env.reset()
print("方策の観測:", observation["observation"][0].tolist())
plt.imshow(env.render())
plt.axis("off")
plt.show()
observation["action"] = torch.tensor([1])  # 0=停止、1=東、2=北東、3=北、…
transition = env.step(observation)["next"]
print("報酬の内訳:", {name: value.item() for name, value in transition["reward_terms"].items()})
env.close()

## Step 9 — 報酬を変更して学習・評価する

`train`は毎回新しい方策を学習します。ここでは同じseed42で標準条件と変更条件を試します。
検証・重みの保存はAPIが行います。`run.checkpoint`が検証で選んだ方策です。
自分の新しい実験の最終評価には、完成例とは別のseed340000を使います。

In [ ]:
run = train(my_base, WORK_DIR / datetime.now().strftime("baseline_%H%M%S_%f"), seed=42, **options)
reward_run = train(my_task, WORK_DIR / datetime.now().strftime("reward_weights_%H%M%S_%f"), seed=42, **options)
report = run.evaluate(episodes=evaluation_episodes, seed=340000)
reward_report = reward_run.evaluate(episodes=evaluation_episodes, seed=340000)

display(pd.DataFrame([
    {"条件": name, "成功率": value["successes"] / value["episode_count"], "最終距離 [m]": value["mean_distance_m"]}
    for name, value in [("標準", report), ("変更", reward_report)]
]))
reward_run.video(WORK_DIR / "reward_weights.mp4", seed=250000)
display(Video(str(WORK_DIR / "reward_weights.mp4"), embed=True, width=640))

## Step 10 — 複数の学習seedで比較する

1 seedで実行方法を確認したら、独立した比較実験として両条件を複数seedで新規学習します。
`compare`が条件とseedの組を実行し、CSVと集計JSONを保存します。
本学習では、2条件×3 seedの学習を実行します。完成例を読むだけなら、この長い再学習は不要です。

In [ ]:
comparison_dir = WORK_DIR / datetime.now().strftime("comparison_%H%M%S_%f")
comparison = compare(
    {"baseline": my_base, "changed": my_task}, comparison_dir,
    seeds=seeds, evaluation_episodes=evaluation_episodes, evaluation_seed=340000,
    **options,
)
display(pd.read_csv(comparison_dir / "comparison.csv"))
display(pd.DataFrame(comparison["summary"]).T)

my_results = pd.read_csv(comparison_dir / "comparison.csv")
display(my_results.groupby("condition")[["success_rate", "mean_distance_m"]].agg(["mean", "std"]))


自分の実験でも、最終値だけでなく検証曲線と同じseed同士の差を確認します。
下の図と元データは`WORK_DIR`に保存されます。`demo`の図は描画方法を確かめるためのものです。

In [ ]:
curve_rows = []
for record in comparison["records"]:
    directory = Path(record["checkpoint"]).parent
    validations = json.loads((directory / "validations.json").read_text())
    for entry in validations:
        value = entry["report"]
        curve_rows.append({
            "condition": record["condition"], "seed": record["seed"],
            "steps": entry["rollout"] * options["num_envs"] * options["horizon"],
            "success_rate": value["successes"] / value["episode_count"],
        })
my_curves = pd.DataFrame(curve_rows)
my_curves.to_csv(WORK_DIR / "student_learning_curves.csv", index=False)
for (condition, seed), group in my_curves.groupby(["condition", "seed"]):
    plt.plot(group["steps"], group["success_rate"], label=f"{condition}, seed {seed}")
plt.xlabel("Training samples")
plt.ylabel("Validation success rate")
plt.ylim(0, 1)
plt.legend()
plt.savefig(WORK_DIR / "student_learning_curves.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for seed, group in my_results.groupby("seed"):
    pair = group.set_index("condition").loc[["baseline", "changed"]]
    for ax, metric in zip(axes, ("success_rate", "mean_distance_m")):
        ax.plot([0, 1], pair[metric], "o-", label=f"Seed {seed}")
        ax.set_xticks([0, 1], ["Baseline", "Changed"])
        ax.set_ylabel(metric)
axes[0].set_ylim(0, 1)
axes[1].legend()
fig.tight_layout()
fig.savefig(WORK_DIR / "student_paired_results.png")
plt.show()

## Step 11 — 発展：独自の報酬関数を追加する

完成例は重み1項目の変更でした。次は、目標の10 cm以内で円柱が速く動くほど減点する関数を作ります。
`before`と`after`は前後の状態、戻り値は並列の世界ごとに1つの数値です。
`weight=-2.0`は減点の強さ、`per_second=True`は時間に応じて積分する指定です。

`%%writefile`で小さなPythonファイルに保存すると、重みと一緒に関数の定義も保存できます。
この実験も観測・行動は標準と同じにします。

In [ ]:
%%writefile {WORK_DIR}/my_reward.py
from dataclasses import replace
from hexapod_transport_rl import CylinderPushTask, RewardTerm

def near_goal_speed(before, after):
    return after.cargo_speed.square() * (after.goal_distance < 0.10)

baseline = CylinderPushTask()
TASK = replace(baseline, rewards={
    **baseline.rewards,
    "near_goal_speed": RewardTerm(near_goal_speed, weight=-2.0, per_second=True),
})

In [ ]:
custom_task = load_task_file(WORK_DIR / "my_reward.py")
custom_task = replace(custom_task, config=my_base.config)
custom_run = train(custom_task, WORK_DIR / datetime.now().strftime("custom_reward_%H%M%S_%f"), seed=42, **options)
custom_report = custom_run.evaluate(episodes=evaluation_episodes, seed=340000)
print("成功:", custom_report["successes"], "/", custom_report["episode_count"])
print("最終距離 [m]:", custom_report["mean_distance_m"])

## Step 12 — 発展：観測・行動を変える

次の課題も`train`へ渡せます。これらを調べるときは、報酬を標準のまま固定します。
観測・行動の意味が変わるので、新しい方策を学習します。
報酬を変更した結果と同時にまとめず、それぞれの仮説を立てて比較します。

In [ ]:
observation_task = replace(my_base, observations=(
    ObservationTerm("cargo_to_robot", scale=0.5),
    ObservationTerm("cargo_to_goal", scale=0.5),
    "robot_heading",
))
action_task = replace(my_base, actions=DiscreteDirections(count=4))
continuous_task = replace(my_base, actions=ContinuousVelocity())

## Step 13 — 結果を考察して保存する

次の内容をMarkdownセルへ書きます。

1. 結果を見る前の仮説と、変更した1項目。
2. 全seedの成功率・最終距離、平均・標準偏差、学習曲線。
3. 仮説と合った点／合わなかった点。根拠になる図・表・動画。
4. この学習予算・3 seed・初期状態の範囲では言えないこと。
5. 次に確かめたい1項目。

たとえば、近づくが押さない、目標を通り過ぎる、seedで優劣が逆になる、といった場合には、
[実験の進め方](https://github.com/hashimoto-robotics-lab/hexapod-transport-rl/blob/main/docs/experiment-guide.md)の対応表を使います。
総報酬が増えたことだけを理由に、良い方策になったとは結論しません。

実験前の仮説は`experiment_plan.json`に残ります。下の`REPORT`を自分の文章に書き換え、考察も保存します。
`task.json`に設定、`task_sources/`に独自関数、`environment.json`に依存バージョンが残ります。
Colab終了前に実験ディレクトリごと保存してください。

In [ ]:
REPORT = f"""# 自分の比較実験

仮説（結果を見る前）：{MY_HYPOTHESIS}
変更した項目：goal_progress、20 → {PROGRESS_WEIGHT}
実行プロファイル：{PROFILE}
学習seed：{seeds}、最終評価seed：340000

## 結果の事実
全seedの数値と平均・標準偏差をここに書く。
根拠のファイル：comparison_*/comparison.csv、student_learning_curves.png、student_paired_results.png

## 仮説と照らし合わせた解釈
数値・曲線・動画のどこから、何を考えたかを書く。

## この実験の限界
学習予算、seed数、初期状態の範囲などを書く。

## 次に確かめる1項目
値と理由をここに書く。
"""
(WORK_DIR / "report.md").write_text(REPORT)
print(REPORT)

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK_DIR), "zip", root_dir=WORK_DIR.parent, base_dir=WORK_DIR.name)
print("実験結果ZIP:", archive)

DOWNLOAD_RESULTS = False  # 手元に保存するときは True
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)

APIの詳しい項目は`docs/student-api.md`、完成例の再実行条件は`examples/student/reward_study.py`にあります。
実測値・図・動画・評価用重みは`results/reward_study/`へまとめています。
Colabには既存のPyTorchを使い、追加の依存ライブラリを準備します。